# 06 -- Visualization

`qwenmoe/viz/` is what every Manim scene in this project builds on:
`common.py::capture_trace` sources real numbers from a real checkpoint,
and the `scenes_*.py` files turn those real numbers into animations.

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## `capture_trace` (`common.py`) -- the one place every Manim scene gets real numbers from
Loads a checkpoint, encodes a prompt with the exact tokenizer it was
trained with, and runs `forward_with_trace` -- never illustrative or
fabricated numbers.

In [2]:
from qwenmoe.viz.common import capture_trace
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint

checkpoint_path = "runs/smoke_test/checkpoints/step_1000.pt"
bundle = capture_trace(checkpoint_path, "ROMEO:")
tokenizer = build_tokenizer_from_meta(load_checkpoint(checkpoint_path)["tokenizer_meta"])

print("token_ids:", bundle.token_ids, "->", repr(tokenizer.decode(bundle.token_ids)))
print("embeddings shape:", tuple(bundle.embeddings.shape))
print("logits shape:", tuple(bundle.logits.shape), " probs shape:", tuple(bundle.probs.shape))

top_id = bundle.probs.argmax().item()
print(f"top predicted next character: {tokenizer.decode([top_id])!r}  (p={bundle.probs[top_id].item():.4f})")

token_ids: [30, 27, 25, 17, 27, 10] -> 'ROMEO:'
embeddings shape: (6, 64)
logits shape: (6, 65)  probs shape: (65,)
top predicted next character: '\n'  (p=0.8828)


### One block's `LayerTrace` in detail

In [3]:
layer0 = bundle.layers[0]
has_pos_trace = layer0.rope_angles_or_alibi_bias is not None
print("rope_angles_or_alibi_bias shape:", tuple(layer0.rope_angles_or_alibi_bias.shape) if has_pos_trace else None)
print("attn_scores_post_softmax shape:", tuple(layer0.attn_scores_post_softmax.shape))
print("router_probs shape:", tuple(layer0.router_probs.shape))
print("topk_indices:")
print(layer0.topk_indices)
print("expert_outputs present for experts:", list(layer0.expert_outputs.keys()),
      "of", layer0.router_probs.shape[-1], "total routed experts")

rope_angles_or_alibi_bias shape: (6, 16)
attn_scores_post_softmax shape: (4, 6, 6)
router_probs shape: (6, 4)
topk_indices:
tensor([[0, 3],
        [3, 0],
        [3, 1],
        [3, 1],
        [3, 2],
        [3, 1]])
expert_outputs present for experts: [0, 1, 2, 3] of 4 total routed experts


## Manim scenes (`scenes_*.py`) -- module inventory
Five **formal** scenes (task 13, animate real architecture math) plus two
**training-observability** scenes (task 14, read a JSON-lines log
instead of a checkpoint) are documented and run via `scripts/render_viz.py`:

In [4]:
scenes = {
    "embedding": ("qwenmoe/viz/scenes_embedding.py", "EmbeddingPositionalScene"),
    "attention": ("qwenmoe/viz/scenes_attention.py", "AttentionScene"),
    "moe": ("qwenmoe/viz/scenes_moe.py", "MoERoutingScene"),
    "norm_act": ("qwenmoe/viz/scenes_norm_act.py", "NormActivationScene"),
    "output": ("qwenmoe/viz/scenes_output.py", "OutputProjectionScene"),
    "loss_curve": ("qwenmoe/viz/scenes_training.py", "LossCurveScene"),
    "expert_utilization": ("qwenmoe/viz/scenes_training.py", "ExpertUtilizationScene"),
}
for name, (path, scene_cls) in scenes.items():
    print(f"{name:20s} uv run manim -pql {path} {scene_cls}")

print()
print("Plus 9 supplementary 'preview' scenes (scenes_task{N}_preview.py, N in")
print("3,5,6,7,8,9,10,11,15) -- manually render-verified only, no dedicated pytest")
print("coverage, one per early task that had no formal Manim deliverable of its own.")

embedding            uv run manim -pql qwenmoe/viz/scenes_embedding.py EmbeddingPositionalScene
attention            uv run manim -pql qwenmoe/viz/scenes_attention.py AttentionScene
moe                  uv run manim -pql qwenmoe/viz/scenes_moe.py MoERoutingScene
norm_act             uv run manim -pql qwenmoe/viz/scenes_norm_act.py NormActivationScene
output               uv run manim -pql qwenmoe/viz/scenes_output.py OutputProjectionScene
loss_curve           uv run manim -pql qwenmoe/viz/scenes_training.py LossCurveScene
expert_utilization   uv run manim -pql qwenmoe/viz/scenes_training.py ExpertUtilizationScene

Plus 9 supplementary 'preview' scenes (scenes_task{N}_preview.py, N in
3,5,6,7,8,9,10,11,15) -- manually render-verified only, no dedicated pytest
coverage, one per early task that had no formal Manim deliverable of its own.


## Actually rendering one scene
This shells out to the real `manim` CLI (exactly what `scripts/render_viz.py`
does), at the fastest quality (`-ql`, 480p15). Takes on the order of ten
seconds; produces a real .mp4 built from the real `bundle` captured above.

In [5]:
import subprocess
from IPython.display import Video

command = ["uv", "run", "manim", "-ql", "qwenmoe/viz/scenes_attention.py", "AttentionScene"]
print("$", " ".join(command))
subprocess.run(command, check=True)

video_path = max(Path("media/videos/scenes_attention/480p15").glob("*.mp4"), key=lambda p: p.stat().st_mtime)
print("rendered:", video_path)
Video(str(video_path), embed=True)

$ uv run manim -ql qwenmoe/viz/scenes_attention.py AttentionScene
Manim Community v0.21.0

[08/31/26 20:28:53] INFO     Animation 0 : Using cached     ]8;id=4080658;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080659;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             2118183852_3072573587_22313245                     
                             7)                                                 
                    INFO     Animation 1 : Using cached     ]8;id=4080664;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080665;file:///Users/sjoseph/Documents/Study/training-my-moe/

[08/31/26 20:28:54] INFO     Animation 2 : Using cached     ]8;id=4080670;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080671;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_2363707152_270979622                     
                             6)                                                 


                    INFO     Animation 3 : Using cached     ]8;id=4080676;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080677;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_760396969_2452053134                     
                             )                                                  
                    INFO     Animation 4 : Using cached     ]8;id=4080682;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080683;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
         

[08/31/26 20:28:55] INFO     Animation 5 : Using cached     ]8;id=4080688;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080689;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_1189990274_644669181                     
                             )                                                  


                    INFO     Animation 6 : Using cached     ]8;id=4080694;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080695;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_170846798_1083736363                     
                             )                                                  
                    INFO     Animation 7 : Using cached     ]8;id=4080700;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080701;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
         

[08/31/26 20:28:56] INFO     Animation 8 : Using cached     ]8;id=4080706;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080707;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_3873173991_154446525                     
                             5)                                                 
                    INFO     Animation 9 : Using cached     ]8;id=4080712;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080713;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
         

... (9 outputs left out)

[08/31/26 20:29:01] INFO     Animation 24 : Using cached    ]8;id=4080802;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080803;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_170846798_162364529)                     
                    INFO     Animation 25 : Using cached    ]8;id=4080808;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080809;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
         

                    INFO     Animation 26 : Using cached    ]8;id=4080814;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080815;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_2645707024_291090282                     
                             1)                                                 
[08/31/26 20:29:02] INFO     Animation 27 : Using cached    ]8;id=4080820;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080821;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
         

[08/31/26 20:29:03] INFO     Animation 28 : Using cached    ]8;id=4080826;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080827;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
                             data (hash :                                       
                             366519484_3873173991_258813617                     
                             )                                                  
                    INFO     Animation 29 : Using cached    ]8;id=4080832;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=4080833;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/renderer/cairo_renderer.py#92\92]8;;\
         

[08/31/26 20:29:04] INFO     Combining to Movie file.   ]8;id=4080840;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=4080841;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#952\952]8;;\
                    INFO                               ]8;id=4080847;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=4080848;file:///Users/sjoseph/Documents/Study/training-my-moe/.venv/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#1103\1103]8;;\
                             File ready at                                      
                             '/Users/sjoseph/Documents                          
                             /Study/training-my-moe/me                          
      